# Suite BRG — The Python bridge

`Bridges.Python` carries expressions between mbse-expressions' Python dialect and the Python language's syntax
trees, both ways, tree to tree. Each case writes terms as syntax nodes, whose printed Python is the dialect's own
render, and reads syntax nodes as terms. Here the bridge's reading of source is also checked against the dialect's
own `parse`, and the literals it decodes against Python's, and the functions it writes run, which only Python can do.

In [ ]:
import ast
import math

from mbse.Expressions.Dialects.Python import Evaluators, Expressions as E
from mbse.Expressions.Framework import Terms as F
from mbse.Programs.Bridges import Python as B
from mbse.Programs.Framework.Errors import TranspileError
from mbse.Programs.Python import Python314
from mbse.Programs.Python import Syntax as P
from support import raises

Py = P.LANGUAGE.Builders


def printed(term):
    return Python314.print(B.module(term)).rstrip("\n")


def read(source):
    return B.term_of_module(Python314.parse(source))


def fails(source, message):
    with raises(TranspileError, match=message):
        read(source)

## BRG-01 · Every term of the dialect writes the Python the dialect renders, and reads back the same

In [ ]:
x, y, z = E.name("x"), E.name("y"), E.name("z")
TERMS = [
    E.constant(1), E.name("this"), E.attribute(x, "age"), E.subscript(x, "key"), E.index(x, E.name("i")),
    E.call("np.add", 1, 2), E.compare("not in", x, E.name("s")), E.boolop("and", E.boolop("and", x, y), z),
    E.boolop("or", E.boolop("and", x, y), E.boolop("or", z, E.name("w"))), E.binop("**", E.unaryop("-", x), 2),
    E.binop("-", x, E.binop("-", y, z)), E.unaryop("not", E.compare("==", x, 1)), E.ifexp(x, 1, 2),
    E.ifexp(E.ifexp(x, 1, 2), 3, 4), E.call("all", E.generator("p", E.name("ports"), E.name("p"))),
    E.generator("p", E.name("ports"), E.attribute(E.name("p"), "pin"), E.compare(">", E.name("p"), 0), x),
    E.let_("n", 3, E.binop("*", E.name("n"), E.name("n"))),
    E.import_("numpy", E.call("np.greater_equal", x, 18), "np"),
    E.import_("os.path", E.call("os.path.join", "a", "b")),
    E.importfrom("math", "sqrt", E.importfrom("math", "pi", E.call("sqrt", E.name("PI")), "PI")),
]
for term in TERMS:
    assert printed(term) == E.render(term), (printed(term), E.render(term))
    assert F.same(read(printed(term)), term), printed(term)
assert Python314.print(B.expression(TERMS[5])) == "np.add(1, 2)"
with raises(TranspileError, match="an import can only enclose the whole expression"):
    B.expression(TERMS[-1])
with raises(TranspileError, match="an import can only enclose the whole expression"):
    B.module(E.call("f", E.import_("os", x)))
print(len(TERMS), "terms")

## BRG-02 · Constants: a native value is the literal `repr` writes, a negative number a negation, and back

In [ ]:
for value, text in [
    (True, "True"), (False, "False"), (0, "0"), (-1, "-1"), (2 ** 70, "1180591620717411303424"), (1.5, "1.5"),
    (-0.0, "-0.0"), (1e16, "1e+16"), (1e-05, "1e-05"), (math.inf, "float('inf')"), (-math.inf, "-float('inf')"),
    (math.nan, "float('nan')"), ("a'b\"c\n", "'a\\'b\"c\\n'"), ("é😀", "'é😀'"), (b"\x00\xff", "b'\\x00\\xff'"),
    ("", "''"),
]:
    term = E.constant(value)
    assert printed(term) == text == E.render(term), (printed(term), text)
    assert E.render(read(text)) == text, E.render(read(text))
print("ok")

## BRG-03 · Python source reads into terms as the dialect parses it: parentheses, chains, lets, generators, imports

In [ ]:
for source, rendered in [
    ("this.age >= 18 and this.email != ''", "this.age >= 18 and this.email != ''"),
    ("a and b and c or d", "a and b and c or d"),
    ("x['key'] + x[0] - x[i]", "x['key'] + x[0] - x[i]"),
    ("all(p.pin > 0 for p in ports if p.used)", "all(p.pin > 0 for p in ports if p.used)"),
    ("(lambda n: n * n)(3)", "(lambda n: n * n)(3)"),
    ("import numpy as np\nnp.greater_equal(x, 18)", "import numpy as np\nnp.greater_equal(x, 18)"),
    ("import os, json\njson.dumps(os.sep)", "import os\nimport json\njson.dumps(os.sep)"),
    ("from math import sqrt, pi as PI\nsqrt(PI)", "from math import sqrt\nfrom math import pi as PI\nsqrt(PI)"),
    ("-1 + -2.5 - (-x) ** 2", "-1 + -2.5 - (-x) ** 2"),
    ("not a if b else ~c", "not a if b else ~c"),
    ("'a' 'b' + b'c' b'd'", "'ab' + b'cd'"),
    ("f(-0.0, 1e16, 1e-05, 2 ** 70, float('nan'))", "f(-0.0, 1e+16, 1e-05, 2 ** 70, float('nan'))"),
    ("x // y % z << 1 | 3 & 4 ^ 5", "x // y % z << 1 | 3 & 4 ^ 5"),
    ("a in b and a not in c", "a in b and a not in c"),
    ("# a comment\nimport json\njson.dumps(v)  # and another", "import json\njson.dumps(v)"),
    ("((a))[('k')]", "a['k']"),
]:
    term = read(source)
    assert E.render(term) == rendered, E.render(term)
    assert F.same(term, E.parse(source)), source  # as the dialect's own parse reads it
    assert F.same(read(printed(term)), term), printed(term)
assert E.render(B.term(Python314.parse("x + 1").body[0].value)) == "x + 1"
print("ok")

## BRG-04 · Literals are decoded as Python decodes them: numbers, prefixes, quotes and escapes

In [ ]:
for spelling, value in [
    ("True", True), ("0x_1F", 31), ("0o17", 15), ("0b1_0", 2), ("1_000", 1000), ("1.5e-3", 0.0015), ("1.", 1.0),
    (".5", 0.5), ("1_0.5", 10.5), ("'a\\'b'", "a'b"), ('"\\x41\\101\\u00e9\\U0001F600"', "AAé😀"),
    ("'\\a\\b\\f\\n\\r\\t\\v\\\\\\\"'", "\a\b\f\n\r\t\v\\\""), ("'\\q\\8'", "\\q\\8"), ("'\\777'", "ǿ"),
    ("'\\0'", "\0"), ("'''tri\\\nple'''", "triple"), ('"""x"""', "x"), ("r'\\n'", "\\n"), ("U'x'", "x"),
    ("b'\\777\\x7f'", b"\xff\x7f"), ("b'\\0\\u'", b"\0\\u"), ("rb'\\x'", b"\\x"), ("Rb'x'", b"x"),
]:
    assert B.decode(spelling) == value and type(B.decode(spelling)) is type(value), (spelling, B.decode(spelling))
    assert ast.literal_eval(spelling) == value, spelling  # as Python's own reading
print("ok")

## BRG-05 · A rule becomes a function, its imports first, that returns what the dialect evaluates

In [ ]:
rule = E.boolop("and", E.compare(">=", E.name("age"), 18), E.compare(">", E.call("len", E.name("email")), 0))
function = B.function_("is_contactable", ["age", "email"], rule)
assert Python314.print(function) == "def is_contactable(age, email):\n    return age >= 18 and len(email) > 0"
namespace = {}
exec(Python314.print(function), namespace)  # the function, run by Python itself
results = []
for age, email in [(20, "a@b"), (17, "a@b"), (30, "")]:
    results.append(Evaluators.OfAny(rule, {"age": age, "email": email}))
    assert namespace["is_contactable"](age, email) == results[-1]
assert results == [True, False, False]
distance = E.importfrom("math", "sqrt", E.call("sqrt", E.binop("+", E.binop("**", x, 2), E.binop("**", y, 2))))
assert Python314.print(B.function_("distance", ["x", "y"], distance)) == (
    "def distance(x, y):\n    from math import sqrt\n    return sqrt(x ** 2 + y ** 2)")
print("ok")

## BRG-06 · What the dialect cannot hold fails, at the syntax node's path

In [ ]:
for source, message in [
    ("a is not None", "body[0].value.comparisons[0]: the operator 'is not' has no counterpart in the Python dialect"),
    ("a @ b", "body[0].value: the operator '@' has no counterpart in the Python dialect"),
    ("a < b < c", "body[0].value: a comparison has one operator"),
    ("f(x=1)", "body[0].value.keywords[0]: keyword arguments are not supported"),
    ("x[1:2]", "body[0].value.slice: slices are not supported"),
    ("x[1, 2]", "body[0].value.slice: slices are not supported"),
    ("(lambda a, b: a)(1)", "body[0].value: a let binds one name"),
    ("(lambda a: a)(1, 2)", "body[0].value: a let binds one name"),
    ("(lambda p, /, a: a)(1)", "body[0].value: a let binds one name"),
    ("(lambda a, *r: a)(1)", "body[0].value: a let binds one name"),
    ("(lambda a, *, k: a)(1)", "body[0].value: a let binds one name"),
    ("(lambda a, **k: a)(1)", "body[0].value: a let binds one name"),
    ("(lambda a=1: a)(2)", "body[0].value: a let binds one name"),
    ("(x for x in a for y in b)", "body[0].value: a generator has one for, over a name"),
    ("(x async for x in a)", "body[0].value: a generator has one for, over a name"),
    ("(x for x, y in a)", "body[0].value: a generator has one for, over a name"),
    ("None", "body[0].value: only native constants are supported"),
    ("...", "body[0].value: only native constants are supported"),
    ("1j", "body[0].value: only native constants are supported"),
    ("'\\N{BULLET}'", "body[0].value: a \\N{...} escape is not supported"),
    ("'\\U00110000'", "body[0].value: an escape beyond U+10FFFF"),
    ("f'{x}'", "body[0].value: JoinedStr has no counterpart in the Python dialect"),
    ("'a' f'{x}'", "body[0].value.values[1]: JoinedStr has no counterpart in the Python dialect"),
    ("'a' b'b'", "body[0].value: bytes and strings cannot be joined"),
    ("[1, 2]", "body[0].value: List has no counterpart in the Python dialect"),
    ("x = 1", "the module must end with an expression"),
    ("import a", "the module must end with an expression"),
    ("import a\nb = 1\nc", "body[1]: only imports may precede the expression"),
    ("from . import a\na", "body[0]: only imports may precede the expression"),
    ("from m import *\na", "body[0]: import * binds names that cannot be known"),
]:
    fails(source, message)
for statement in [Py.Import().is_lazy(True).add_names(Py.Alias().name(Py.DottedName().add_names("a"))),
                  Py.ImportFrom().is_lazy(True).module(Py.DottedName().add_names("m")).add_names(
                      Py.Alias().name(Py.DottedName().add_names("a")))]:
    lazy = Py.Module().body([statement, Py.Expr().value(Py.Name().id("a"))]).create()
    with raises(TranspileError, match="body[0]: only imports may precede the expression"):  # a lazy import
        B.term_of_module(lazy)
print("ok")